# Two-model RoBERTa classifier

This notebook fine-tunes two independent `roberta-base` models: one for `aspectCategory` and one for `polarity`. Both models use the same multilabel-stratified 80:20 train/test split by review ID, preventing duplicate review text from leaking between partitions.

In [1]:
# Run once if these packages are not installed.
!uv pip install -q pandas scikit-learn iterative-stratification datasets transformers accelerate torch

In [2]:
from pathlib import Path
import gc
import random
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

/home/kami/Projects/NLP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
repo_root = Path.cwd()
if not (repo_root / "data" / "contest2_train.csv").exists():
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / "data" / "contest2_train.csv")
required_columns = {"id", "text", "aspectCategory", "polarity"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
df = df.dropna(subset=list(required_columns)).copy()
df["id"] = df["id"].astype(str)
df["text"] = df["text"].astype(str)

print(f"Rows: {len(df):,}; unique reviews: {df['id'].nunique():,}")
display(df.head())
display(pd.crosstab(df["aspectCategory"], df["polarity"], margins=True))

Rows: 3,156; unique reviews: 2,584


,id,text,aspectCategory,polarity
0,3121,But the staff was so horrible to us.,service,negative
1,2777,"To be completely fair, the only redeeming fact...",food,positive
2,2777,"To be completely fair, the only redeeming fact...",anecdotes/miscellaneous,negative
3,1634,"The food is uniformly exceptional, with a very...",food,positive
4,2534,Where Gabriela personaly greets you and recomm...,service,positive


polarity,conflict,negative,neutral,positive,All
aspectCategory,,,,,
ambience,41,78,21,228,368
anecdotes/miscellaneous,24,176,285,471,956
food,57,182,69,743,1051
price,15,100,8,152,275
service,30,179,15,282,506
All,167,715,398,1876,3156


## Stratified 80:20 train/test split

Stratification is performed at review level over every observed category–polarity combination. The resulting review IDs are then used to select the original rows for the two single-label models.

In [4]:
aspects = sorted(df["aspectCategory"].unique())
polarities = sorted(df["polarity"].unique())
joint_labels = [(aspect, polarity) for aspect in aspects for polarity in polarities]
joint_label2id = {label: index for index, label in enumerate(joint_labels)}

review_rows = []
for review_id, group in df.groupby("id", sort=False):
    if group["text"].nunique() != 1:
        raise ValueError(f"Review {review_id} contains more than one text.")
    targets = np.zeros(len(joint_labels), dtype=np.int8)
    for pair in zip(group["aspectCategory"], group["polarity"]):
        targets[joint_label2id[pair]] = 1
    review_rows.append({"id": review_id, "targets": targets.tolist()})
reviews = pd.DataFrame(review_rows)
targets = np.asarray(reviews["targets"].tolist())

splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(reviews[["id"]], targets))
train_reviews = reviews.iloc[train_idx].reset_index(drop=True)
test_reviews = reviews.iloc[test_idx].reset_index(drop=True)

train_ids = set(train_reviews["id"])
test_ids = set(test_reviews["id"])
assert train_ids.isdisjoint(test_ids)

train_df = df[df["id"].isin(train_ids)].reset_index(drop=True)
test_df = df[df["id"].isin(test_ids)].reset_index(drop=True)

for name, ids, rows in [("train", train_ids, train_df), ("test", test_ids, test_df)]:
    print(f"{name:>5}: {len(ids):4d} reviews, {len(rows):4d} labeled rows ({len(ids) / len(reviews):.1%})")

train: 2074 reviews, 2525 labeled rows (80.3%)
 test:  510 reviews,  631 labeled rows (19.7%)


In [5]:
# Verify the category and polarity distributions in every partition.
distribution_rows = []
for split_name, split_df in [("all", df), ("train", train_df), ("test", test_df)]:
    for column in ["aspectCategory", "polarity"]:
        for label, fraction in split_df[column].value_counts(normalize=True).items():
            distribution_rows.append({
                "split": split_name, "target": column, "label": label, "fraction": fraction
            })
distribution = pd.DataFrame(distribution_rows)
display(distribution.pivot(index=["target", "label"], columns="split", values="fraction").style.format("{:.2%}"))

## Tokenization and training helpers

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def make_inference_dataset(frame):
    dataset = Dataset.from_pandas(frame[["text"]], preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )

def make_dataset(frame, target_column, label2id):
    model_frame = pd.DataFrame({
        "text": frame["text"],
        "labels": frame[target_column].map(label2id).astype(int),
    })
    dataset = Dataset.from_pandas(model_frame, preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )

def metric_function(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }

def train_classifier(target_column, class_names):
    label2id = {label: index for index, label in enumerate(class_names)}
    id2label = {index: label for label, index in label2id.items()}
    datasets = {
        "train": make_dataset(train_df, target_column, label2id),
        "test": make_dataset(test_df, target_column, label2id),
    }
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(class_names), label2id=label2id, id2label=id2label
    )
    output_dir = repo_root / "artifacts" / f"roberta_{target_column}"
    args = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=32,
        num_train_epochs=4,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="no",
        save_strategy="no",
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=datasets["train"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=metric_function,
    )
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    return trainer, datasets, label2id, id2label

contest_df = pd.read_csv(repo_root / "data" / "contest2_test.csv")
contest_dataset = make_inference_dataset(contest_df)

Map: 100%|██████████| 461/461 [00:00<00:00, 59332.11 examples/s]


## Model 1: aspect-category classifier

In [8]:
aspect_trainer, aspect_datasets, aspect_label2id, aspect_id2label = train_classifier(
    "aspectCategory", aspects
)
aspect_output = aspect_trainer.predict(aspect_datasets["test"])
aspect_predictions = np.argmax(aspect_output.predictions, axis=-1)
test_aspects = [aspect_id2label[index] for index in aspect_predictions]
print(classification_report(
    aspect_output.label_ids, aspect_predictions, target_names=aspects, zero_division=0
))
contest_aspect_indices = np.argmax(
    aspect_trainer.predict(contest_dataset).predictions, axis=-1
)
contest_aspects = [aspect_id2label[index] for index in contest_aspect_indices]

# Release the first model before the second model is created.
del aspect_output, aspect_predictions, aspect_trainer, aspect_datasets
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Map: 100%|██████████| 631/631 [00:00<00:00, 86782.50 examples/s]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.784800


                         precision    recall  f1-score   support

               ambience       0.66      0.55      0.60        74
anecdotes/miscellaneous       0.86      0.83      0.84       191
                   food       0.76      0.79      0.77       210
                  price       0.61      0.65      0.63        55
                service       0.63      0.68      0.65       101

               accuracy                           0.74       631
              macro avg       0.70      0.70      0.70       631
           weighted avg       0.75      0.74      0.74       631



## Model 2: polarity classifier

In [9]:
polarity_trainer, polarity_datasets, polarity_label2id, polarity_id2label = train_classifier(
    "polarity", polarities
)
polarity_output = polarity_trainer.predict(polarity_datasets["test"])
polarity_predictions = np.argmax(polarity_output.predictions, axis=-1)
test_polarities = [polarity_id2label[index] for index in polarity_predictions]
print(classification_report(
    polarity_output.label_ids, polarity_predictions, target_names=polarities, zero_division=0
))
contest_polarity_indices = np.argmax(
    polarity_trainer.predict(contest_dataset).predictions, axis=-1
)
contest_polarities = [polarity_id2label[index] for index in contest_polarity_indices]

# All predictions are plain Python lists now, so the second model can be released too.
del polarity_output, polarity_predictions, polarity_trainer, polarity_datasets
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Map: 100%|██████████| 631/631 [00:00<00:00, 57436.27 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.635700


              precision    recall  f1-score   support

    conflict       0.29      0.36      0.32        33
    negative       0.78      0.73      0.75       143
     neutral       0.67      0.56      0.61        80
    positive       0.88      0.91      0.89       375

    accuracy                           0.80       631
   macro avg       0.66      0.64      0.65       631
weighted avg       0.80      0.80      0.80       631



## Predict the unlabeled contest set

Each model produces one label per input sentence. The two predictions are combined into the required category and polarity columns.

In [10]:
predictions_df = contest_df.copy()
predictions_df["aspectCategory"] = contest_aspects
predictions_df["polarity"] = contest_polarities
prediction_path = repo_root / "artifacts" / "roberta_two_model_predictions.csv"
prediction_path.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(prediction_path, index=False)
print(f"Saved predictions to {prediction_path}")
display(predictions_df.head(10))

Saved predictions to /home/kami/Projects/NLP/Contest2/artifacts/roberta_two_model_predictions.csv


,id,text,aspectCategory,polarity
0,899,Food and service was okay.,service,neutral
1,1349,"great for a romantic evening, or a fun evening...",anecdotes/miscellaneous,positive
2,934,The lamb meat was under-cooked and EXTRMELY CH...,food,negative
3,2199,The best pad thai i've ever had.,food,positive
4,188,"Over time, the food quality has decreased subs...",food,positive
5,1748,Great pizza and fantastic service.,service,positive
6,949,We were seated outside and the waiter spilled ...,service,negative
7,390,i would just ask for no oil next time.,food,negative
8,885,While this is a pretty place in that overly cu...,food,conflict
9,1428,I went there for lunch and it was not as good ...,anecdotes/miscellaneous,negative


In [12]:
# Evaluate against gold labels from the held-out 20%, not the unlabeled contest set.
test_gold_path = repo_root / "artifacts" / "roberta_two_model_test_gold.csv"
test_prediction_path = repo_root / "artifacts" / "roberta_two_model_test_predictions.csv"
test_gold_path.parent.mkdir(parents=True, exist_ok=True)
test_df[["id", "text", "aspectCategory", "polarity"]].to_csv(
    test_gold_path, index=False
)
pd.DataFrame({
    "id": test_df["id"],
    "aspectCategory": test_aspects,
    "polarity": test_polarities,
}).to_csv(test_prediction_path, index=False)

subprocess.run(
    [
        sys.executable,
        str(repo_root / "scripts" / "evaluate.py"),
        str(test_gold_path),
        str(test_prediction_path),
    ],
    check=True,
)

=== CLASSIFICATION : ASPECT ===
                class name  precision  recall  F1-score  support
0                     food      0.000   0.000     0.000        1
1                    price      0.000   0.000     0.000        0
2                  service      0.000   0.000     0.000        0
3                 ambience      0.000   0.000     0.000        0
4  anecdotes/miscellaneous      0.000   0.000     0.000        0
5                MACRO AVG      0.000   0.000     0.000        1
6                MICRO AVG      0.000   0.000     0.000        1 

=== CLASSIFICATION : SENTIMENT ===
  class name  precision  recall  F1-score  support
0   positive      0.000   0.000     0.000        0
1   negative      0.000   0.000     0.000        0
2    neutral      0.015   1.000     0.029        1
3   conflict      0.000   0.000     0.000        0
4  MACRO AVG      0.004   0.250     0.007        1
5  MICRO AVG      0.002   1.000     0.004        1 

=== CLASSIFICATION : OVERALL ===
              preci